In [ ]:
from standardbots import models, StandardBotsRobot

MAX_ITER = 1000
MAX_FRAMES = 3

sdk = StandardBotsRobot(
    url='https://mybot.sb.app',
    token='token',
    robot_kind=StandardBotsRobot.RobotKind.Live,
)



In [ ]:
with sdk.connection():
    res = sdk.camera.data.get_camera_stream()

try:
    res.ok()

    n_frames = 0
    buffer = b""
    for i, chunk in enumerate(res.response.stream(1024)):
        buffer += chunk
        # Search for the start (0xffd8) and end (0xffd9) of the JPEG frame
        a = buffer.find(b"\xff\xd8")
        b = buffer.find(b"\xff\xd9")
        if a != -1 and b != -1:
            jpg = buffer[a : b + 2]  # Extract the JPEG image data
            buffer = buffer[
                b + 2 :
            ]  # Remove the processed frame from the buffer

            with open(f"frame_{n_frames}.jpeg", "wb") as fp:
                fp.write(jpg)

            print("Found a frame")
            n_frames += 1

        if i > MAX_ITER or n_frames + 1 > MAX_FRAMES:
            break
finally:
    # Always close the connection
    res.response.release_conn()
